In [4]:
import pandas as pd
import numpy as np
from env_RTB_dataaugument import OfflineRTBEnv

def generate_rtb_history_data(num_rows=1000, feature_dim=5):
    """
    Tạo dữ liệu lịch sử đấu giá mẫu.
    - features: [hour_norm, location_score, user_score, device_type, random_noise]
    - bid_1: Giá thầu cao nhất
    - bid_2: Giá thầu cao thứ hai
    """
    np.random.seed(42)
    data = []
    
    for i in range(num_rows):
        # 1. Tạo các đặc trưng ngữ cảnh (Contextual Features)
        hour = np.random.randint(0, 24)
        location_id = np.random.randint(1, 11) # 10 địa điểm khác nhau
        user_affinity = np.random.random()    # Độ quan tâm của user
        device_type = np.random.randint(0, 2) # 0: Mobile, 1: PC
        
        # Chuẩn hóa features về khoảng [0, 1] để mạng Neural dễ học
        feat_vec = [
            hour / 23.0,               # Giờ
            location_id / 10.0,        # Địa điểm
            user_affinity,             # User
            float(device_type),        # Thiết bị
            np.random.random()         # Nhiễu ngẫu nhiên
        ]
        
        # 2. Giả lập logic giá Bid (Intrinsic Value)
        # Giả sử: PC (device=1) và Giờ cao điểm (18-22h) bid sẽ cao hơn
        base_price = 30 + (location_id * 3) + (user_affinity * 20)
        if 18 <= hour <= 22:
            base_price += 25
        if device_type == 1:
            base_price += 15
            
        # Tạo giá bid_1 với phân phối Log-Normal (sát với thực tế RTB)
        bid_1 = np.random.lognormal(mean=np.log(base_price), sigma=0.15)
        
        # bid_2 thường thấp hơn bid_1 khoảng 10-30%
        bid_2 = bid_1 * np.random.uniform(0.7, 0.9)
        
        # 3. Giả lập kịch bản "Dữ liệu bị che" (Censored)
        # Khoảng 15% dữ liệu là đấu giá thất bại (không có ai bid trên giá sàn cũ)
        if np.random.random() < 0.15:
            bid_1 = np.nan
            bid_2 = np.nan
            
        data.append({
            'features': feat_vec,
            'bid_1': bid_1,
            'bid_2': bid_2
        })
        
    return pd.DataFrame(data)

# --- Sử dụng thử ---
# 1. Tạo dữ liệu
df_history = generate_rtb_history_data(num_rows=2000)

# 2. Khởi tạo môi trường với dữ liệu vừa tạo
env = OfflineRTBEnv(dataframe=df_history, target_revenue=5000)

# 3. Reset thử môi trường
obs, _ = env.reset()
print(f"Trạng thái khởi tạo (t, b, x_t): \n{obs}")

Trạng thái khởi tạo (t, b, x_t): 
[1.00000000e+02 5.00000000e+03 5.21739125e-01 3.00000012e-01
 3.29542130e-01 1.00000000e+00 5.06371439e-01]


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
import torch


df_train = generate_rtb_history_data(num_rows=5000)
env = OfflineRTBEnv(dataframe=df_train, target_revenue=2000)

ppo_config = {
    "policy": "MlpPolicy",
    "env": env,
    "learning_rate": 1e-4,      
    "n_steps": 256,              
    "batch_size": 64,            
    "n_epochs": 10,              
    "gamma": 0.99,              
    "gae_lambda": 0.95,
    "clip_range": 0.2,           
    "ent_coef": 0.01,            
    "verbose": 1,
    "device": "auto"         
}

model = PPO(**ppo_config)

# 3. Huấn luyện Agent
print("Đang bắt đầu huấn luyện RL-armRP Agent...")
model.learn(total_timesteps=20000)

# 4. Lưu mô hình
model.save("ppo_rtb_reserve_price_kdd25")

print("Huấn luyện hoàn tất!")

c:\Users\admin\AppData\Local\pypoetry\Cache\virtualenvs\finrl-8jNU-FJQ-py3.12\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Đang bắt đầu huấn luyện RL-armRP Agent...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | 7.18e+03 |
| time/              |          |
|    fps             | 445      |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 256      |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 100           |
|    ep_rew_mean          | 6.94e+03      |
| time/                   |               |
|    fps                  | 372           |
|    iterations           | 2             |
|    time_elapsed         | 1             |
|    total_timesteps      | 512           |
| train/                  |               |
|    approx_kl            | 5.7399273e-05 |
|    clip_fraction        | 0           

In [6]:
def evaluate_agent(model, env, num_episodes=10):
    all_revenues = []
    success_deals = 0
    total_steps = 0
    
    for _ in range(num_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = env.step(action)
            
            all_revenues.append(info['revenue'])
            if info['revenue'] > 0:
                success_deals += 1
            total_steps += 1
            
    ar = np.mean(all_revenues)
    dr = success_deals / total_steps
    
    print("\n--- KẾT QUẢ ĐÁNH GIÁ (Section 5.1.3) ---")
    print(f"Average Revenue (AR): {ar:.4f}")
    print(f"Deal Rate (DR): {dr:.2%}")
    return ar, dr

# Chạy đánh giá
evaluate_agent(model, env)


--- KẾT QUẢ ĐÁNH GIÁ (Section 5.1.3) ---
Average Revenue (AR): 48.8657
Deal Rate (DR): 86.00%


(np.float64(48.86571037856006), 0.86)